# 04: Análises e gráficos

**TC3 · PosTech FIAP Data Analytics**  
Responsável: Caio Bosnic

---

Responde as sete perguntas do enunciado em cima da Gold e grava um gráfico por
pergunta em `results/graficos/`, que é o material da apresentação executiva.

**Entrada:** as 16 tabelas de `gold/`, geradas por `03_gold.ipynb`.  
**Saída:** `results/graficos/p1..p7.png` e as tabelas de apoio.

As mesmas contas existem em SQL, em `sql/perguntas/`, para rodar no Athena.
Notebook e SQL devolvem o mesmo número: é a contraprova de um contra o outro.


## A armadilha do denominador

Antes de qualquer percentual, uma regra que vale para a pesquisa inteira.

Vários blocos do questionário são **condicionais**: só aparecem para quem se
encaixa. O bloco de estratégia de IA só é mostrado a gestor, o de maturidade
só a quem trabalha com engenharia de dados. Somado a isso, `NULL` em `qtd_*`
quer dizer **não viu a pergunta** e `0` quer dizer **viu e não marcou nada**.

Percentual sobre os 14.002 quando a base real são 2.593 gestores não erra por
pouco: inverte a leitura. Por isso cada gráfico daqui carrega a base ao lado.


In [ ]:
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


def achar_raiz():
    p = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(p, "glue", "gold", "job_gold.py")):
            return p
        pai = os.path.dirname(p)
        if pai == p:
            break
        p = pai
    raise RuntimeError("Não achei a raiz do projeto a partir de " + os.getcwd())


RAIZ = achar_raiz()
GOLD = os.environ.get("TC3_PATH_GOLD", os.path.join(RAIZ, "data", "gold"))
SAIDA = os.path.join(RAIZ, "results", "graficos")
os.makedirs(SAIDA, exist_ok=True)


def num(v, casas=1, prefixo="", sufixo=""):
    """
    Número no padrão brasileiro: ponto no milhar, vírgula no decimal.

    O `format` do Python faz o contrário (1,234.5) e isso vazaria para o rótulo
    do gráfico, que vai para a apresentação. O § é separador temporário, para
    trocar os dois símbolos sem um sobrescrever o outro.
    """
    s = f"{v:,.{casas}f}".replace(",", "§").replace(".", ",").replace("§", ".")
    return f"{prefixo}{s}{sufixo}"


def ler(tabela):
    """Lê uma tabela da Gold com pandas. O volume cabe em memória: 14.002 linhas."""
    return pd.read_parquet(os.path.join(GOLD, "gold_dw_" + tabela))


fato = ler("fat_profissionais")
print("fato:", num(len(fato), 0), "linhas")
print("gráficos vão para:", SAIDA)


### Padrão visual dos gráficos

As cores são as mesmas do dashboard, para o material executivo ficar coerente:
azul `#2E6BA8` e dourado `#C08A3E`. O par foi conferido para daltonismo
(separação ΔE 22,9 em protanopia, bem acima do piso de 8) e todo gráfico traz
**rótulo direto em cada barra**, porque o dourado fica abaixo de 3:1 de
contraste com o fundo branco e o número escrito resolve isso.

Para as três edições a escala é sequencial, do claro ao escuro, porque edição
é ordem e não identidade.


In [ ]:
AZUL, DOURADO = "#2E6BA8", "#C08A3E"
EDICOES = ["#A8C4E0", "#6E9BC9", "#2E6BA8"]   # sequencial: 2023 -> 2025
TINTA, TINTA2 = "#3D3B36", "#5C5A54"

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": "#D8D6CF", "axes.labelcolor": TINTA2, "axes.titlecolor": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.grid": False, "figure.facecolor": "white", "axes.facecolor": "white",
})


def moldura(ax, titulo, subtitulo=None):
    """Título em duas linhas, eixos discretos. O subtítulo é onde a base aparece."""
    ax.set_title(titulo, loc="left", fontsize=12, fontweight="bold", pad=18 if subtitulo else 8)
    if subtitulo:
        ax.text(0, 1.02, subtitulo, transform=ax.transAxes, fontsize=9, color=TINTA2)
    return ax


def salvar(fig, nome):
    caminho = os.path.join(SAIDA, nome + ".png")
    fig.savefig(caminho)
    plt.close(fig)
    print("gravado:", os.path.relpath(caminho, RAIZ))


## Os denominadores, explícitos

A tabela que sustenta todos os percentuais deste notebook. Ela também está no
dashboard, na tela Fontes e método.


In [ ]:
bases = pd.DataFrame({
    "respondentes": fato.groupby("edicao").size(),
    "empregados": fato[fato.sk_cargo.notna()].groupby("edicao").size(),
    "gestores": fato[fato.eh_gestor == True].groupby("edicao").size(),
})
bases.loc["total"] = bases.sum()
bases


---

## P1. Como está estruturado o mercado brasileiro de Dados?


In [ ]:
cargo = ler("dim_cargo")
# A dimensão nomeia a ausência em vez de deixar a FK nula. Para "cargo mais
# frequente" esses três membros não são cargo: fora da base dá os 10.173
# empregados na área de dados, que é a base do enunciado.
NAO_E_CARGO = ["Gestor, sem cargo técnico", "Não declarado", "Fora da área de dados"]
emp = fato.merge(cargo, on="sk_cargo").query("rotulo not in @NAO_E_CARGO")
top = emp.rotulo.value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.barh(top.index, top.values, color=AZUL, height=0.62)
for y, v in enumerate(top.values):
    ax.text(v * 1.01, y, num(v, 0), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, top.max() * 1.14)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Os 10 cargos mais frequentes",
        "base " + num(len(emp), 0) + " empregados na área de dados, três edições")
salvar(fig, "p1_mercado")
print("\n3 maiores concentram " + num(100*top.tail(3).sum()/len(emp), 1, sufixo="% da base"))


## P2. Quais perfis são mais valorizados?

Salário médio por senioridade. A escala é sequencial porque senioridade é
ordem, não identidade.


In [ ]:
sen = ler("dim_senioridade")
NIVEIS = ["Júnior", "Pleno", "Sênior", "Especialista / Staff+"]
sal = (fato.merge(sen, on="sk_senioridade")
            .query("rotulo in @NIVEIS and salario_estimado == salario_estimado")
            .groupby("rotulo").salario_estimado.mean().reindex(NIVEIS))

fig, ax = plt.subplots(figsize=(7, 4))
rampa = ["#A8C4E0", "#6E9BC9", "#3E7CB5", "#1F3A5F"]
ax.bar(range(len(sal)), sal.values, color=rampa, width=0.58)
for x, v in enumerate(sal.values):
    ax.text(x, v * 1.02, num(v, 0, "R$ "), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(range(len(sal)))
ax.set_xticklabels([n.replace(" / ", "\n") for n in sal.index], fontsize=9)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_ylim(0, sal.max() * 1.16)
moldura(ax, "Salário médio por senioridade",
        "o salto de júnior para pleno é o mais rentável: +" + num(100*(sal.iloc[1]/sal.iloc[0]-1), 1, sufixo="%"))
salvar(fig, "p2_perfis")


## P3. Qual é o cenário de diversidade de gênero?


In [ ]:
gen = ler("dim_genero")
g = fato.merge(gen, on="sk_genero_valor")

# Denominador: TODOS os respondentes, não só quem marcou Feminino ou Masculino.
# "Participação feminina no mercado" é fatia do total; tirar do denominador quem
# respondeu "Outro" ou "Prefere não informar" infla o número. São 68 pessoas, o
# efeito é de 0,1 ponto, mas a definição é a que vale.
fem = g.groupby("edicao").rotulo.apply(lambda s: 100 * (s == "Feminino").mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(fem)), fem.values, color=EDICOES, width=0.5)
for x, v in enumerate(fem.values):
    ax.text(x, v + 0.4, num(v, 1, sufixo="%"), ha="center", fontsize=10, color=TINTA)
ax.set_xticks(range(len(fem)))
ax.set_xticklabels(fem.index, fontsize=9)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_ylim(0, fem.max() * 1.22)
moldura(ax, "Participação feminina, por edição",
        "cai a cada edição · base " + num(len(g), 0) + " respondentes")
salvar(fig, "p3_diversidade")


## P4. Quais tecnologias têm maior adoção?

Múltipla escolha: a soma passa de 100% de propósito. O denominador é quem
**viu o bloco de linguagens**, não os 14.002.


In [ ]:
opc = ler("dim_opcao")
bridge = ler("bridge_respondente_opcao")


def adocao(grupo, n=10):
    """% de adoção sobre quem respondeu o bloco, que é a base correta."""
    sks = opc.loc[opc.grupo == grupo, ["sk_opcao", "opcao_rotulo"]]
    m = bridge.merge(sks, on="sk_opcao")
    base = m.sk_respondente.nunique()
    cont = m.groupby("opcao_rotulo").sk_respondente.nunique().sort_values(ascending=False).head(n)
    return 100 * cont / base, base


ling, base_ling = adocao("linguagens", 10)
ling = ling.sort_values()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.barh(ling.index, ling.values, color=AZUL, height=0.62)
for y, v in enumerate(ling.values):
    ax.text(v + 1.2, y, num(v, 1, sufixo="%"), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, 104)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Linguagens mais usadas",
        "% sobre " + num(base_ling, 0) + " que responderam o bloco · múltipla escolha")
salvar(fig, "p4_tecnologias")


## P5. Qual é o índice de adoção de IA e seu impacto?

**O gráfico central do trabalho.** Duas populações distintas, dois
denominadores diferentes:

- **uso pessoal**: quem não é gestor e viu o bloco de uso de IA
- **prioridade da empresa**: só gestores, que são os únicos a ver o bloco de estratégia

Juntar as duas num denominador só é exatamente o erro que a tela Fontes e
método existe para evitar.


In [ ]:
pessoal = fato[fato.usa_ia_generativa.notna()]
empresa = fato[fato.ia_prioridade_alta.notna()]

usa = 100 * pessoal.groupby("edicao").usa_ia_generativa.mean()
prio = 100 * empresa.groupby("edicao").ia_prioridade_alta.mean()

fig, ax = plt.subplots(figsize=(8, 4.6))
x = range(len(usa))
ax.bar([i - 0.19 for i in x], usa.values, width=0.34, color=AZUL, label="pessoas que usam IA")
ax.bar([i + 0.19 for i in x], prio.values, width=0.34, color=DOURADO, label="empresas que priorizam IA")
for i, (a, b) in enumerate(zip(usa.values, prio.values)):
    ax.text(i - 0.19, a + 1.5, num(a, 1, sufixo="%"), ha="center", fontsize=9, color=TINTA)
    ax.text(i + 0.19, b + 1.5, num(b, 1, sufixo="%"), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(list(x))
ax.set_xticklabels(usa.index, fontsize=9)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_ylim(0, 112)
ax.legend(frameon=False, fontsize=9, ncol=2, labelcolor=TINTA2,
          loc="lower left", bbox_to_anchor=(0, -0.22))
moldura(ax, "A adoção individual corre à frente da estratégia",
        "bases distintas: " + num(len(pessoal), 0) + " não gestores e " + num(len(empresa), 0) + " gestores")
salvar(fig, "p5_ia")
print("\ndistância em 2025: " + num(usa.iloc[-1]-prio.iloc[-1], 1, sufixo=" pontos percentuais"))


## P6. Existem diferenças por região, senioridade ou modelo de trabalho?


In [ ]:
geo = ler("dim_geografia")
reg = (fato.merge(geo, on="sk_geografia")
            .query("regiao != 'Não declarado' and salario_estimado == salario_estimado")
            .groupby("regiao").salario_estimado.mean().sort_values())

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.barh(reg.index, reg.values, color=AZUL, height=0.58)
for y, v in enumerate(reg.values):
    ax.text(v * 1.01, y, num(v, 0, "R$ "), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, reg.max() * 1.16)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Salário médio por região",
        num(100*(reg.max()/reg.min()-1), 0, sufixo="% separa a região mais cara da mais barata"))
salvar(fig, "p6_regiao")


## P7. Quais oportunidades e desafios?

As barreiras à IA, apontadas por quem decide. Base: gestores que marcaram ao
menos uma barreira.


In [ ]:
barr, base_barr = adocao("barreiras_ia_generativa", 7)
barr = barr.sort_values()

fig, ax = plt.subplots(figsize=(8.4, 4))
cores = [DOURADO if v >= barr.nlargest(2).min() else AZUL for v in barr.values]
ax.barh(barr.index, barr.values, color=cores, height=0.6)
for y, v in enumerate(barr.values):
    ax.text(v + 0.6, y, num(v, 1, sufixo="%"), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, barr.max() * 1.18)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "O que trava a adoção de IA",
        "em dourado, as duas maiores · base " + num(base_barr, 0) + " gestores")
salvar(fig, "p7_oportunidades")
print("\nas duas maiores somam " + num(barr.nlargest(2).sum(), 1, sufixo="% e são de capital humano"))


---

## Gráficos complementares

Os sete acima cobrem uma pergunta cada. Estes abrem os recortes que a
apresentação usa para sustentar cada achado.


### Formação: de onde vem o profissional


In [ ]:
form = ler("dim_formacao")
nf = (fato.merge(form, on="sk_nivel_ensino")
           .query("rotulo != 'Prefere não informar'")
           .groupby(["ordem", "rotulo"]).size().reset_index(name="n").sort_values("ordem"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(nf)), nf.n, color=AZUL, width=0.6)
for x, v in enumerate(nf.n):
    ax.text(x, v * 1.02, num(v, 0), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(range(len(nf)))
ax.set_xticklabels([r.replace(" de ", "\n").replace(" / ", "\n") for r in nf.rotulo], fontsize=8)
ax.set_yticks([]); ax.set_ylim(0, nf.n.max() * 1.14)
ax.spines["left"].set_visible(False)
pos = nf[nf.ordem >= 4].n.sum() / nf.n.sum()
moldura(ax, "Nível de formação", num(100*pos, 1, sufixo="% tem pós-graduação ou acima"))
salvar(fig, "c1_formacao")


### Salário por cargo: onde o dinheiro está


In [ ]:
sc = (emp.query("salario_estimado == salario_estimado")
         .groupby("rotulo").agg(n=("salario_estimado", "size"), media=("salario_estimado", "mean"))
         .query("n >= 50").sort_values("media").tail(10))

fig, ax = plt.subplots(figsize=(8.4, 4.4))
ax.barh(sc.index, sc.media, color=AZUL, height=0.62)
for y, v in enumerate(sc.media):
    ax.text(v * 1.01, y, num(v, 0, "R$ "), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, sc.media.max() * 1.17); ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Salário médio por cargo", "só cargos com 50 respondentes ou mais")
salvar(fig, "c2_salario_cargo")


### Diversidade: o gap salarial cresce com a senioridade


In [ ]:
g2 = (fato.merge(gen, on="sk_genero_valor").merge(sen, on="sk_senioridade", suffixes=("_g", "_s"))
           .query("rotulo_g in ['Feminino', 'Masculino'] and rotulo_s in @NIVEIS")
           .query("salario_estimado == salario_estimado"))
piv = g2.pivot_table(index="rotulo_s", columns="rotulo_g", values="salario_estimado", aggfunc="mean").reindex(NIVEIS)
gap = 100 * (piv["Masculino"] - piv["Feminino"]) / piv["Masculino"]

fig, ax = plt.subplots(figsize=(7.4, 4))
ax.bar(range(len(gap)), gap.values, color=["#A8C4E0", "#6E9BC9", "#C08A3E", "#1F3A5F"], width=0.56)
for x, v in enumerate(gap.values):
    ax.text(x, v + 0.35, num(v, 1, sufixo="%"), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(range(len(gap)))
ax.set_xticklabels([n.replace(" / ", "\n") for n in gap.index], fontsize=9)
ax.set_yticks([]); ax.set_ylim(0, gap.max() * 1.22)
ax.spines["left"].set_visible(False)
moldura(ax, "Gap salarial por senioridade", "quanto o homem ganha a mais, controlado por nível")
salvar(fig, "c3_gap_senioridade")


### Diversidade: acesso à liderança


In [ ]:
lid = (fato.merge(gen, on="sk_genero_valor").query("eh_gestor == eh_gestor")
            .groupby("rotulo").eh_gestor.mean() * 100).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.6))
cor = [DOURADO if i == "Feminino" else AZUL for i in lid.index]
ax.barh(lid.index, lid.values, color=cor, height=0.56)
for y, v in enumerate(lid.values):
    ax.text(v + 0.4, y, num(v, 1, sufixo="%"), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, lid.max() * 1.2); ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Quem chega a gestor, por gênero", "em dourado, o recorte feminino")
salvar(fig, "c4_lideranca")


### Tecnologias: clouds e bancos


In [ ]:
cl, base_cl = adocao("clouds", 7)
cl = cl.sort_values()
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.barh(cl.index, cl.values, color=AZUL, height=0.58)
for y, v in enumerate(cl.values):
    ax.text(v + 0.8, y, num(v, 1, sufixo="%"), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, cl.max() * 1.2); ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Clouds utilizadas",
        "a soma passa de 100%: multi-cloud é a norma · base " + num(base_cl, 0))
salvar(fig, "c5_clouds")


### IA: quem paga pela ferramenta

Sobre a base de uso pessoal, que exclui gestor. A empresa banca em menos de
um quinto dos casos: é o gap da tese, visto pelo bolso.


In [ ]:
qp, base_qp = adocao("uso_ia_pessoal", 6)
qp = qp.sort_values()
fig, ax = plt.subplots(figsize=(8.6, 3.6))
cor = [DOURADO if "empresa" in i.lower() else AZUL for i in qp.index]
ax.barh(qp.index, qp.values, color=cor, height=0.58)
for y, v in enumerate(qp.values):
    ax.text(v + 0.8, y, num(v, 1, sufixo="%"), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, qp.max() * 1.22); ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Quem paga pela ferramenta de IA",
        "em dourado, quando a empresa paga · base " + num(base_qp, 0) + " no uso pessoal")
salvar(fig, "c6_quem_paga")


### Região e modelo: o modelo pesa mais que o mapa


In [ ]:
mod = ler("dim_modelo_trabalho")
mt = (fato.merge(mod, on="sk_modelo_trabalho")
           .query("rotulo != 'Não declarado' and salario_estimado == salario_estimado")
           .groupby("rotulo").salario_estimado.mean().sort_values())

fig, ax = plt.subplots(figsize=(7.6, 3.4))
cor = [DOURADO if "remoto" in i.lower() else AZUL for i in mt.index]
ax.barh(mt.index, mt.values, color=cor, height=0.56)
for y, v in enumerate(mt.values):
    ax.text(v * 1.01, y, num(v, 0, "R$ "), va="center", fontsize=9, color=TINTA)
ax.set_xlim(0, mt.max() * 1.18); ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
moldura(ax, "Salário médio por modelo de trabalho", "em dourado, o 100% remoto")
salvar(fig, "c7_modelo_trabalho")


### A pirâmide de senioridade

O achado que sustenta a recomendação de recompor a base. Só os três níveis
que existem nas três edições, e só linhas comparáveis entre anos: sem esse
corte, um nível que nasceu em 2025 entra na conta e inverte a leitura.


In [ ]:
pir = (fato.query("serie_comparavel").merge(sen, on="sk_senioridade")
            .query("rotulo in ['Júnior', 'Pleno', 'Sênior']"))
tab = pir.pivot_table(index="edicao", columns="rotulo", values="sk_respondente", aggfunc="count")
tab = tab[["Júnior", "Pleno", "Sênior"]]
pct = 100 * tab.div(tab.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 4.2))
larg = 0.26
for k, nivel in enumerate(pct.columns):
    x = [i + (k - 1) * larg for i in range(len(pct))]
    ax.bar(x, pct[nivel], width=larg, color=EDICOES[k], label=nivel)
    for xi, v in zip(x, pct[nivel]):
        ax.text(xi, v + 0.7, num(v, 1, sufixo="%"), ha="center", fontsize=8, color=TINTA)
ax.set_xticks(range(len(pct))); ax.set_xticklabels(pct.index, fontsize=9)
ax.set_yticks([]); ax.set_ylim(0, 50)
ax.spines["left"].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=3, labelcolor=TINTA2, loc="lower left", bbox_to_anchor=(0, -0.2))
moldura(ax, "A pirâmide envelheceu",
        "base " + num(len(pir), 0) + " nos três níveis comparáveis entre as edições")
salvar(fig, "c8_piramide")
print("\njúnior:", " -> ".join(num(v, 1, sufixo="%") for v in pct["Júnior"]))


### Validação contra fonte externa

Oito referências públicas conferidas na fonte primária. Três comparam o mesmo
indicador e é o que este gráfico mostra: número nosso contra número deles.


In [ ]:
import textwrap

bm = ler("dim_benchmark").query("uso == 'VALIDAR' and nosso_valor == nosso_valor")
bm = bm.assign(rot=bm.indicador.str.replace("Salário médio, ", "", regex=False))

# Percentual e salário não cabem no mesmo eixo. Tentar resolver com escala
# logarítmica só espreme os dois grupos nas pontas: são duas medidas de
# natureza diferente, então são dois painéis, cada um linear.
grupos = [("%", "Indicadores de adoção, em %"), ("R$/mês", "Salário médio, em R$")]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1]})

for ax, (unid, titulo) in zip(axes, grupos):
    d = bm[bm.unidade == unid].sort_values("nosso_valor")
    casas = 1 if unid == "%" else 0
    y = range(len(d))
    ax.hlines(y, d.valor, d.nosso_valor, color="#D8D6CF", lw=2, zorder=1)
    ax.scatter(d.valor, y, s=80, color=DOURADO, zorder=2, label="fonte externa")
    ax.scatter(d.nosso_valor, y, s=80, color=AZUL, zorder=2, label="nosso número")
    for i, r in enumerate(d.itertuples()):
        esq, dir_ = sorted((r.valor, r.nosso_valor))
        ax.text(esq, i + 0.16, num(esq, casas), ha="right", fontsize=8, color=TINTA2)
        ax.text(dir_, i + 0.16, num(dir_, casas), ha="left", fontsize=8, color=TINTA)
    ax.set_yticks(list(y))
    ax.set_yticklabels([textwrap.fill(r, 26) for r in d.rot], fontsize=9)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    largura = max(d.nosso_valor.max(), d.valor.max()) - min(d.nosso_valor.min(), d.valor.min())
    ax.set_xlim(min(d.nosso_valor.min(), d.valor.min()) - largura * 0.45,
                max(d.nosso_valor.max(), d.valor.max()) + largura * 0.45)
    ax.set_ylim(-0.6, len(d) - 0.2)
    ax.set_title(titulo, loc="left", fontsize=10, color=TINTA2, pad=6)

axes[0].legend(frameon=False, fontsize=9, ncol=2, labelcolor=TINTA2,
               loc="lower left", bbox_to_anchor=(0, -0.3))
fig.suptitle("O nosso número contra a fonte externa", x=0.005, y=1.1,
             ha="left", fontsize=12, fontweight="bold", color=TINTA)
fig.text(0.005, 1.0, num(len(bm), 0) + " das " + num(len(ler("dim_benchmark")), 0)
         + " referências externas comparam o mesmo indicador; as outras delimitam ou complementam",
         fontsize=9, color=TINTA2)
salvar(fig, "c9_benchmark")


---

## Conferência

Os sete PNG em `results/graficos/` são o material da apresentação executiva.
Os números de cada um batem com os CSV de `results/`, que saíram do Athena
pelas consultas de `sql/perguntas/`: são dois motores diferentes chegando ao
mesmo lugar.


In [ ]:
gerados = sorted(f for f in os.listdir(SAIDA) if f.endswith(".png"))
print(f"{len(gerados)} gráficos\n")
for g in gerados:
    kb = os.path.getsize(os.path.join(SAIDA, g)) / 1024
    print(f"  {g:26s} {kb:6.0f} KB")

assert len(gerados) == 16, f"esperado 16 gráficos, vieram {len(gerados)}"
print("\n[OK] 7 por pergunta e 9 complementares")
